In [26]:
import pandas as pd
import glob
import os


datasets = ['tax']

In [27]:
def load_results(dataset):
    """Load all results from CSV files for a given dataset."""
    base_path = "main_csv_results/main_comparison"
    
    results = {
        'dc_profiling': [],
        'norma_document': [],
        'norma_sentence': []
    }
    
    # Load DC Profiling results
    dc_files = glob.glob(os.path.join(base_path, "dc_discovery*.csv"))
    for file in dc_files:
        df = pd.read_csv(file)
        if 'dataset' in df.columns:
            df = df[df['dataset'] == dataset]
        results['dc_profiling'].append(df)
    
    # Load NORMA Document results
    doc_files = glob.glob(os.path.join(base_path, "norma_document*.csv"))
    for file in doc_files:
        df = pd.read_csv(file)
        if 'dataset' in df.columns:
            df = df[df['dataset'] == dataset]
        if 'status' in df.columns:
            df = df[df['status'] == 'success']
        results['norma_document'].append(df)
    
    # Load NORMA Sentence results
    sent_files = glob.glob(os.path.join(base_path, "norma_sentence*.csv"))
    for file in sent_files:
        df = pd.read_csv(file)
        if 'dataset' in df.columns:
            df = df[df['dataset'] == dataset]
        if 'status' in df.columns:
            df = df[df['status'] == 'success']
        results['norma_sentence'].append(df)
    
    # Concatenate results
    for key in results:
        if results[key]:
            results[key] = pd.concat(results[key], ignore_index=True)
        else:
            results[key] = pd.DataFrame()
    
    return results

In [28]:
def create_main_results_table(datasets=['tax'], precision=2):
    """
    Table 1: All approaches with constraint exact metrics and violation aggregate metrics.
    Columns: Constraint Exact (P/R/F1) and Violation Aggregate (P/R/F1)
    """
    rows = []
    
    for dataset in datasets:
        results = load_results(dataset)
        
        # DC Profiling results
        if not results['dc_profiling'].empty:
            for _, result in results['dc_profiling'].iterrows():
                mode = result.get('mode', 'unknown')
                rows.append({
                    'approach': 'DC Profiling',
                    'mode': mode.replace('_', ' ').title(),
                    'dataset': dataset,
                    'constraint_prec': result.get('constraint_exact_precision', None),
                    'constraint_rec': result.get('constraint_exact_recall', None),
                    'constraint_f1': result.get('constraint_exact_f1', None),
                    'violation_prec': result.get('aggregate_violation_precision', None),
                    'violation_rec': result.get('aggregate_violation_recall', None),
                    'violation_f1': result.get('aggregate_violation_f1', None),
                })
        
        # NORMA Document results
        if not results['norma_document'].empty:
            for _, result in results['norma_document'].iterrows():
                mode = result.get('mode', 'Document')
                rows.append({
                    'approach': 'NORMA',
                    'mode': mode.replace('_', ' ').title(),
                    'dataset': dataset,
                    'constraint_prec': result.get('constraint_exact_precision', None),
                    'constraint_rec': result.get('constraint_exact_recall', None),
                    'constraint_f1': result.get('constraint_exact_f1', None),
                    'violation_prec': result.get('aggregate_violation_precision', None),
                    'violation_rec': result.get('aggregate_violation_recall', None),
                    'violation_f1': result.get('aggregate_violation_f1', None),
                })
        
        # NORMA Sentence results
        if not results['norma_sentence'].empty:
            for _, result in results['norma_sentence'].iterrows():
                mode = result.get('mode', 'Sentence')
                rows.append({
                    'approach': 'NORMA',
                    'mode': mode.replace('_', ' ').title(),
                    'dataset': dataset,
                    'constraint_prec': result.get('constraint_exact_precision', None),
                    'constraint_rec': result.get('constraint_exact_recall', None),
                    'constraint_f1': result.get('constraint_exact_f1', None),
                    'violation_prec': result.get('aggregate_violation_precision', None),
                    'violation_rec': result.get('aggregate_violation_recall', None),
                    'violation_f1': result.get('aggregate_violation_f1', None),
                })
    
    # Count rows per dataset
    dataset_row_counts = {}
    for row in rows:
        ds = row['dataset']
        dataset_row_counts[ds] = dataset_row_counts.get(ds, 0) + 1
    
    # Find best F1 scores per dataset
    best_constraint_f1 = {}
    best_violation_f1 = {}
    for row in rows:
        ds = row['dataset']
        if ds not in best_constraint_f1:
            best_constraint_f1[ds] = -1
            best_violation_f1[ds] = -1
        if row['constraint_f1'] is not None and row['constraint_f1'] > best_constraint_f1[ds]:
            best_constraint_f1[ds] = row['constraint_f1']
        if row['violation_f1'] is not None and row['violation_f1'] > best_violation_f1[ds]:
            best_violation_f1[ds] = row['violation_f1']
    
    # Generate LaTeX table
    latex = []
    latex.append(r"\begin{table*}[ht]")
    latex.append(r"\centering")
    latex.append(r"\small")
    latex.append(r"\begin{tabular}{lllcccccc}")
    latex.append(r"\toprule")
    latex.append(r"\textbf{Dataset} & \textbf{Approach} & \textbf{Mode} & \multicolumn{3}{c}{\textbf{Constraint Exact}} & \multicolumn{3}{c}{\textbf{Violation Aggregate}} \\")
    latex.append(r"\cmidrule(lr){4-6} \cmidrule(lr){7-9}")
    latex.append(r" & & & \textbf{P} & \textbf{R} & \textbf{F1} & \textbf{P} & \textbf{R} & \textbf{F1} \\")
    latex.append(r"\midrule")
    
    dataset_first_row = {}
    for i, row in enumerate(rows):
        ds = row['dataset']
        if ds not in dataset_first_row:
            dataset_first_row[ds] = i
    
    def fmt(val, is_best=False):
        if val is None:
            return "-"
        formatted = f"{val:.{precision}f}"
        if is_best and val > 0:
            return r"\textbf{" + formatted + "}"
        return formatted
    
    current_approach = None
    for i, row in enumerate(rows):
        ds = row['dataset']
        
        # Dataset column with multirow
        if i == dataset_first_row[ds]:
            dataset_str = r"\multirow{" + str(dataset_row_counts[ds]) + r"}{*}{" + ds + "}"
            current_approach = None
        else:
            dataset_str = ""
        
        # Approach column
        if row['approach'] != current_approach:
            approach_str = row['approach']
            current_approach = row['approach']
        else:
            approach_str = ""
        
        is_best_constraint = (row['constraint_f1'] is not None and 
                              row['constraint_f1'] == best_constraint_f1[ds])
        is_best_violation = (row['violation_f1'] is not None and 
                             row['violation_f1'] == best_violation_f1[ds])
        
        latex.append(
            f"{dataset_str} & {approach_str} & {row['mode']} & "
            f"{fmt(row['constraint_prec'])} & {fmt(row['constraint_rec'])} & {fmt(row['constraint_f1'], is_best_constraint)} & "
            f"{fmt(row['violation_prec'])} & {fmt(row['violation_rec'])} & {fmt(row['violation_f1'], is_best_violation)} \\\\"
        )
        
        if i < len(rows) - 1 and rows[i+1]['dataset'] != ds:
            latex.append(r"\midrule")
    
    latex.append(r"\bottomrule")
    latex.append(r"\end{tabular}")
    latex.append(r"\caption{Main results for all approaches. Best F1 scores per dataset shown in bold.}")
    latex.append(r"\label{tab:main_results}")
    latex.append(r"\end{table*}")
    
    return "\n".join(latex)


def create_sentence_detailed_table(datasets=['tax'], precision=2):
    """
    Table 2: Sentence-based approaches with pairwise metrics.
    Shows pairwise constraint and violation metrics with standard deviations.
    """
    rows = []
    
    for dataset in datasets:
        results = load_results(dataset)
        
        # Only NORMA Sentence results
        if not results['norma_sentence'].empty:
            for _, result in results['norma_sentence'].iterrows():
                mode = result.get('mode', 'Sentence')
                rows.append({
                    'mode': mode.replace('_', ' ').title(),
                    'dataset': dataset,
                    # Pairwise Constraint (Syntactic)
                    'constraint_exact': result.get('pairwise_syntactic_exact_match', None),
                    'constraint_jaccard': result.get('pairwise_syntactic_predicate_jaccard', None),
                    'constraint_jaccard_std': result.get('pairwise_syntactic_predicate_jaccard_std', None),
                    # Pairwise Violation
                    'violation_exact': result.get('pairwise_violation_exact_match', None),
                    'violation_jaccard': result.get('pairwise_violation_jaccard', None),
                    'violation_jaccard_std': result.get('pairwise_violation_jaccard_std', None),
                })
    
    # Generate LaTeX table
    latex = []
    latex.append(r"\begin{table*}[ht]")
    latex.append(r"\centering")
    latex.append(r"\small")
    latex.append(r"\begin{tabular}{llcccccc}")
    latex.append(r"\toprule")
    latex.append(r"\textbf{Dataset} & \textbf{Mode} & \multicolumn{3}{c}{\textbf{Pairwise Constraint}} & \multicolumn{3}{c}{\textbf{Pairwise Violation}} \\")
    latex.append(r"\cmidrule(lr){3-5} \cmidrule(lr){6-8}")
    latex.append(r" & & \textbf{EM} & \textbf{Jaccard} & \textbf{Std} & \textbf{EM} & \textbf{Jaccard} & \textbf{Std} \\")
    latex.append(r"\midrule")
    
    dataset_row_counts = {}
    for row in rows:
        ds = row['dataset']
        dataset_row_counts[ds] = dataset_row_counts.get(ds, 0) + 1
    
    dataset_first_row = {}
    for i, row in enumerate(rows):
        ds = row['dataset']
        if ds not in dataset_first_row:
            dataset_first_row[ds] = i
    
    def fmt(val):
        if val is None:
            return "-"
        return f"{val:.{precision}f}"
    
    for i, row in enumerate(rows):
        ds = row['dataset']
        
        # Dataset column with multirow
        if i == dataset_first_row[ds]:
            dataset_str = r"\multirow{" + str(dataset_row_counts[ds]) + r"}{*}{" + ds + "}"
        else:
            dataset_str = ""
        
        latex.append(
            f"{dataset_str} & {row['mode']} & "
            f"{fmt(row['constraint_exact'])} & {fmt(row['constraint_jaccard'])} & {fmt(row['constraint_jaccard_std'])} & "
            f"{fmt(row['violation_exact'])} & {fmt(row['violation_jaccard'])} & {fmt(row['violation_jaccard_std'])} \\\\"
        )
        
        if i < len(rows) - 1 and rows[i+1]['dataset'] != ds:
            latex.append(r"\midrule")
    
    latex.append(r"\bottomrule")
    latex.append(r"\end{tabular}")
    latex.append(r"\caption{Pairwise metrics for sentence-based approaches. EM = Exact Match (proportion of perfect matches), Jaccard = average Jaccard similarity, Std = standard deviation.}")
    latex.append(r"\label{tab:sentence_pairwise}")
    latex.append(r"\end{table*}")
    
    return "\n".join(latex)

In [29]:
# Generate Table 1: Main results for all approaches
main_table = create_main_results_table(datasets=['tax'], precision=2)
print(main_table)

\begin{table*}[ht]
\centering
\small
\begin{tabular}{lllcccccc}
\toprule
\textbf{Dataset} & \textbf{Approach} & \textbf{Mode} & \multicolumn{3}{c}{\textbf{Constraint Exact}} & \multicolumn{3}{c}{\textbf{Violation Aggregate}} \\
\cmidrule(lr){4-6} \cmidrule(lr){7-9}
 & & & \textbf{P} & \textbf{R} & \textbf{F1} & \textbf{P} & \textbf{R} & \textbf{F1} \\
\midrule
\multirow{5}{*}{tax} & DC Profiling & Exact & 0.20 & 0.17 & 0.18 & 0.00 & 0.02 & 0.01 \\
 &  & Approximate & 0.00 & 0.00 & 0.00 & 0.00 & 1.00 & 0.01 \\
 & NORMA & Document & 0.30 & 0.50 & 0.38 & 1.00 & 0.79 & 0.88 \\
 &  & Sentence Direct & 0.45 & 0.83 & 0.59 & 1.00 & 0.98 & \textbf{0.99} \\
 &  & Sentence Filtered & 0.67 & 0.67 & \textbf{0.67} & 1.00 & 0.79 & 0.88 \\
\bottomrule
\end{tabular}
\caption{Main results for all approaches. Best F1 scores per dataset shown in bold.}
\label{tab:main_results}
\end{table*}


In [30]:
# Generate Table 2: Detailed sentence-based results
sentence_table = create_sentence_detailed_table(datasets=['tax'], precision=2)
print(sentence_table)

\begin{table*}[ht]
\centering
\small
\begin{tabular}{llcccccc}
\toprule
\textbf{Dataset} & \textbf{Mode} & \multicolumn{3}{c}{\textbf{Pairwise Constraint}} & \multicolumn{3}{c}{\textbf{Pairwise Violation}} \\
\cmidrule(lr){3-5} \cmidrule(lr){6-8}
 & & \textbf{EM} & \textbf{Jaccard} & \textbf{Std} & \textbf{EM} & \textbf{Jaccard} & \textbf{Std} \\
\midrule
\multirow{2}{*}{tax} & Sentence Direct & 0.83 & 0.83 & 0.37 & 0.83 & 0.83 & 0.37 \\
 & Sentence Filtered & 0.67 & 0.67 & 0.47 & 0.67 & 0.67 & 0.47 \\
\bottomrule
\end{tabular}
\caption{Pairwise metrics for sentence-based approaches. EM = Exact Match (proportion of perfect matches), Jaccard = average Jaccard similarity, Std = standard deviation.}
\label{tab:sentence_pairwise}
\end{table*}
